In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# load the data
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

# preprocess the data
train.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
test.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# encode categorical variables
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# fill missing values
train.fillna(train.mean(), inplace=True)
test.fillna(test.mean(), inplace=True)

# split the data into training and validation sets
X_train = train.drop(['Survived'], axis=1)
y_train = train['Survived']

# create a pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier())
])

# define the parameter grid
param_grid = {
    'clf__n_estimators': [100, 200, 500],
    'clf__max_depth': [5, 10, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
}

# perform grid search cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

# print the best parameters and score
print('Best parameters:', grid_search.best_params_)
print('Best score:', grid_search.best_score_)

# train the final model
final_model = RandomForestClassifier(n_estimators=grid_search.best_params_['clf__n_estimators'],
                                      max_depth=grid_search.best_params_['clf__max_depth'],
                                      min_samples_split=grid_search.best_params_['clf__min_samples_split'],
                                      min_samples_leaf=grid_search.best_params_['clf__min_samples_leaf'])
final_model.fit(X_train, y_train)

# make predictions on the test set
X_test = test
y_pred = final_model.predict(X_test)

Best parameters: {'clf__max_depth': 20, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 5, 'clf__n_estimators': 500}
Best score: 0.8327788588286987


In [2]:
# save the predictions to a CSV file
output = pd.DataFrame({'PassengerId': pd.read_csv('/kaggle/input/titanic/test.csv')['PassengerId'], 'Survived': y_pred})
output.to_csv('submission.csv', index=False)